# Sigmoid loss for Language-Image Pre-training (SigLIP)

Zhai, Xiaohua, Basil Mustafa, Alexander Kolesnikov, and Lucas Beyer. “Sigmoid Loss for Language Image Pre-Training.” arXiv:2303.15343. Preprint, arXiv, September 27, 2023. https://doi.org/10.48550/arXiv.2303.15343.


In [1]:
import math
from dataclasses import dataclass

import torch
from torch import nn
from datasets import load_dataset
from scipy.spatial import distance
from torch.utils.data import DataLoader
from transformers import AutoProcessor, AutoModel, AutoTokenizer

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# hiperparaméterek
lr = 0.001
n_epochs = 3
batch_size = 32
weight_decay = 0.0001

## Adathalmaz és előfeldolgozás

A SigLIP modelleket a WebLI adathalmazon tanították, amit a Google nem adott közzé (legalábbis eddig), ezért a reprodukáláshoz az `flickr30k` adathalmaz lett kiválasztva.

Az adathalmaz (kép, szöveg) rendezett párokból áll, ahol a szöveg mindegyik képről egy leírást ad, hogy mi történik rajta.

Először, egy angol szövegekből álló modell lett tanítva (SigLIP), majd mind a 100 nyelvet megtartották az adathalmazból és azon egy új modell lett tanítva (mSigLIP).

In [ ]:
ds = load_dataset("AnyModal/flickr30k", split="train")

In [ ]:
ds

In [ ]:
for example in ds:
    break

In [ ]:
example["image"]

In [ ]:
example["alt_text"]

In [ ]:
processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")

In [5]:
# processor.image_processor
# processor.tokenizer

In [ ]:
def collate_fn(examples):
    images, texts = [], []

    for example in examples:
        image, text = example["image"], example["alt_text"]
        images.append(image)
        texts.append(text)

    batch = processor(
        images=images,
        text=texts,
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )
    return batch

In [ ]:
dataloader = DataLoader(ds, batch_size=batch_size, collate_fn=collate_fn)

In [ ]:
for batch in dataloader:
    break

In [ ]:
batch["pixel_values"].shape, batch["input_ids"].shape

(torch.Size([32, 3, 224, 224]), torch.Size([32, 64]))

## Konfiguráció

In [7]:
@dataclass
class TextConfig:
    hidden_size: int = 192
    projection_size: int = 192 # = hidden size
    layer_norm_eps: float = 1e-6
    vocab_size: int = 32000
    max_position_embeddings: int = 64
    num_hidden_layers: int = 3
    intermediate_size: int = 384
    num_hidden_layers: int = 3
    n_heads: int = 12
    attention_dropout: float = 0.0

@dataclass
class VisionConfig:
    hidden_size: int = 192
    intermediate_size: int = 384
    num_hidden_layers: int = 3
    n_heads: int = 3
    layer_norm_eps: float = 1e-6
    attention_dropout: float = 0.0
    num_channels: int = 3
    image_size: int = 224
    patch_size: int = 16

## Modell

Legyen $f(\cdot)$ képi model (image model) és $g(\cdot)$ szöveges model (text model)

A szöveges modell bemente $x \in \mathbb{R}^{b \times s}$, ahol
- $b$ a batch-méret,
- $s$ a szekvencia hossza tokenizálás után.

A képi modell bemente $x \in \mathbb{R}^{b \times c \times h \times w}$, ahol

- $b$ a batch-méret
- $c$ a csatornák száma (pl. RGB esetén $c=3$),
- $h$ a kép magassága pixelekben,
- $w$ a kép szélessége pixelekben.

A szöveges modell és képi modell kimenete ugyanolyan méretű $f(x),g(x) \in \mathbb{R}^{b \times d}$, ahol $b$ a batch-méret és $d$ az embedding dimenziója.

Ahhoz, hogy a két modell kimenete kompatibilis legyen, két módosítás szükséges.

Először, mivel egy szöveges modell kimenete $\mathbb{R}^{b \times s \times d}$ alakú (ahol $b$ a batch-méret, $s$ a szekvencia hossza és $d$ a beágyazás mérete). Ha minden szekvenciába vesszük az utolsó tokent (ami lehet padding token is), akkor minden szekvenciát lehet ábrázolni egy vektorral és a modell megtanul erre a tokenre figyelni és ide kompresszálni az információt. Itt nem használnak semmilyen poolingot.

Másodszor, a képi modell - jelen esetben ViT (vision transformer) - kimenete $\mathbb{R}^{b \times p \times d}$ alakú (ahol $b$ a batch-méret, $p$ a patch-ek száma és $d$ a beágyazás mérete). Ebben az esetben pooling-ot használnak, méghozzá egy többfejű figyelem alapú pooling-ot, ami segítségével a kimenet $\mathbb{R}^{b \times d}$-ra változik. Azaz minden példához egy vektor fog tartozni.

Megjegyzés. A cikkben erről nem láttam információt, ezért a `transformers` könyvtárvban lévő implementációt vettem itt alapul.

In [ ]:
class TextEmbeddings(nn.Module):
    def __init__(self, config: TextConfig) -> None:
        super().__init__()

        self.max_position_embeddings = config.max_position_embeddings

        self.token_embedding = nn.Embedding(config.vocab_size, config.hidden_size)
        self.position_embedding = nn.Embedding(config.max_position_embeddings, config.hidden_size)

        # "explicit" batch processing (also contiguous in memory, exported when serialized)
        self.register_buffer(
            "position_ids", torch.arange(config.max_position_embeddings).expand((1, -1)), persistent=False
        )

    def forward(
        self,
        input_ids: torch.LongTensor,
        position_ids: torch.LongTensor | None = None,
    ) -> torch.FloatTensor:

        seq_length = input_ids.shape[-1] # (batch_size, seq_length)

        assert seq_length <= self.max_position_embeddings, f"seq length must be less than max_position_embeddings, got seq length {seq_length}"

        if position_ids is None:
            position_ids = self.position_ids[:, :seq_length]

        inputs_embeds = self.token_embedding(input_ids)
        position_embeddings = self.position_embedding(position_ids)

        embeddings = inputs_embeds + position_embeddings

        return embeddings

In [11]:
class GELUTanh(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, x: torch.FloatTensor) -> torch.FloatTensor:
        return x * 0.5 * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3.0))))

In [12]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.config = config

        self.activation_fn = GELUTanh()
        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.fc2 = nn.Linear(config.intermediate_size, config.hidden_size)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = self.fc1(hidden_states)
        hidden_states = self.activation_fn(hidden_states)
        hidden_states = self.fc2(hidden_states)
        return hidden_states

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.config = config

        self.n_heads = config.n_heads
        self.head_dim = config.hidden_size // self.n_heads

        assert self.head_dim * self.n_heads == config.hidden_size, "embed_dim must be divisible by num_heads"

        self.scale = self.head_dim**-0.5
        self.dropout = config.attention_dropout

        self.k_proj = nn.Linear(config.hidden_size, config.hidden_size)
        self.v_proj = nn.Linear(config.hidden_size, config.hidden_size)
        self.q_proj = nn.Linear(config.hidden_size, config.hidden_size)
        self.out_proj = nn.Linear(config.hidden_size, config.hidden_size)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:

        batch_size, seq_length, embed_dim = hidden_states.shape

        queries = self.q_proj(hidden_states)
        keys = self.k_proj(hidden_states)
        values = self.v_proj(hidden_states)

        queries = queries.view(batch_size, seq_length, self.n_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(batch_size, seq_length, self.n_heads, self.head_dim).transpose(1, 2)
        values = values.view(batch_size, seq_length, self.n_heads, self.head_dim).transpose(1, 2)

        # (batch_size, n_heads, seq_length, head_dim) @ (batch_size, n_heads, head_dim, seq_length)
        # -> (batch_size, n_heads, seq_length, seq_length)
        attn_weights = torch.matmul(queries, keys.transpose(-1, -2)) * self.scale

        if attention_mask is not None: # (batch_size, 1, seq_length, seq_length)
            attn_weights = attn_weights + attention_mask

        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(queries.dtype)
        attn_weights = nn.functional.dropout(attn_weights, p = self.dropout if self.training else 0.0)

        attn_output = torch.matmul(attn_weights, values)
        attn_output = attn_output.transpose(1, 2).contiguous()

        attn_output = attn_output.reshape(batch_size, seq_length, embed_dim).contiguous()
        attn_output = self.out_proj(attn_output)

        return attn_output

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, config: VisionConfig | TextConfig) -> None:
        super().__init__()

        self.embed_dim = config.hidden_size
        self.layer_norm1 = nn.LayerNorm(self.embed_dim, eps=config.layer_norm_eps)
        self.self_attn = MultiHeadAttention(config)
        self.layer_norm2 = nn.LayerNorm(self.embed_dim, eps=config.layer_norm_eps)
        self.mlp = MLP(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.FloatTensor:

        residual = hidden_states
        hidden_states = self.layer_norm1(hidden_states)
        hidden_states = self.self_attn(hidden_states, attention_mask)
        hidden_states = residual + hidden_states

        residual = hidden_states
        hidden_states = self.layer_norm2(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states

        return hidden_states

In [ ]:
class Encoder(nn.Module):
    def __init__(self, config: TextConfig) -> None:
        super().__init__()

        self.config = config
        self.layers = nn.ModuleList([EncoderLayer(config) for _ in range(config.num_hidden_layers)])

    def forward(
        self,
        inputs_embeds,
        attention_mask: torch.Tensor | None = None,
    ):
        hidden_states = inputs_embeds
        for encoder_layer in self.layers:
            hidden_states = encoder_layer(hidden_states, attention_mask)
        return hidden_states

In [ ]:
def create_attention_mask(input_ids: torch.Tensor, pad_token_id: int = 1) -> torch.Tensor:
    return (input_ids != pad_token_id).long()

In [ ]:
def create_bidirectional_attn_mask(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:

    batch_size, seq_len, _ = hidden_states.shape

    # create 4D boolean attention mask of shape (batch_size, 1, seq_len, seq_len),
    # where a value of True indicates that the element should take part in the
    # attention computation, and False that it should not.

    # (batch, seq_len) -> (batch, 1, 1, seq_len)
    attention_mask = attention_mask[:, None, None, :]

    # (batch, 1, 1, seq_len) -> (batch, 1, seq_len, seq_len)
    attention_mask = attention_mask.expand(batch_size, 1, seq_len, seq_len)

    dtype = hidden_states.dtype
    min_dtype = torch.finfo(dtype).min
    # 0s where the tokens should be taken into account, and -inf otherwise
    mask = torch.where(attention_mask.bool(), torch.tensor(0.0, device=attention_mask.device, dtype=dtype), min_dtype)
    return mask

In [ ]:
class TextModel(nn.Module):

    def __init__(self, config: TextConfig) -> None:
        super().__init__()

        self.config = config
        self.embeddings = TextEmbeddings(config)
        self.encoder = Encoder(config)

        self.final_layer_norm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.head = nn.Linear(config.hidden_size, config.hidden_size)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None
    ):

        input_shape = input_ids.size() # ? shape
        input_ids = input_ids.view(-1, input_shape[-1]) # ? shape

        hidden_states = self.embeddings(input_ids=input_ids, position_ids=position_ids)

        if attention_mask is None:
            attention_mask = create_attention_mask(input_ids)

        attention_mask = create_bidirectional_attn_mask(hidden_states, attention_mask)

        last_hidden_state = self.encoder(inputs_embeds=hidden_states, attention_mask=attention_mask)
        last_hidden_state = self.final_layer_norm(last_hidden_state)

        # model uses the last token's hidden state, which may be padding.
        # ? inherited from CLIP (as that was a decoder)
        pooler_output = last_hidden_state[:, -1, :]
        pooler_output = self.head(pooler_output)

        return last_hidden_state, pooler_output

In [ ]:
class VisionEmbeddings(nn.Module):
    def __init__(self, config: VisionConfig):
        super().__init__()
        self.config = config
        self.embed_dim = config.hidden_size
        self.image_size = config.image_size
        self.patch_size = config.patch_size

        self.patch_embedding = nn.Conv2d(
            in_channels=config.num_channels,
            out_channels=config.hidden_size,
            kernel_size=config.patch_size,
            stride=config.patch_size,
            padding="valid",
        )

        self.num_patches = (self.image_size // self.patch_size) ** 2

        self.position_embedding = nn.Embedding(self.num_patches, config.hidden_size)
        self.register_buffer("position_ids", torch.arange(self.num_patches).expand((1, -1)), persistent=False)

    def interpolate_pos_encoding(self, embeddings: torch.Tensor, height: int, width: int) -> torch.Tensor:
        # TODO leírni https://en.wikipedia.org/wiki/Bicubic_interpolation

        num_patches = embeddings.shape[1]
        num_positions = self.position_embedding.weight.shape[0]

        patch_pos_embed = self.position_embedding.weight.unsqueeze(0)

        dim = embeddings.shape[-1]

        new_height = height // self.patch_size
        new_width = width // self.patch_size

        sqrt_num_positions = int(num_positions**0.5)
        patch_pos_embed = patch_pos_embed.reshape(1, sqrt_num_positions, sqrt_num_positions, dim)
        patch_pos_embed = patch_pos_embed.permute(0, 3, 1, 2)

        patch_pos_embed = nn.functional.interpolate(
            patch_pos_embed,
            size=(new_height, new_width),
            mode="bicubic",
            align_corners=False,
        )

        patch_pos_embed = patch_pos_embed.permute(0, 2, 3, 1).view(1, -1, dim)
        return patch_pos_embed

    def forward(self, pixel_values: torch.FloatTensor, interpolate_pos_encoding: bool = False) -> torch.Tensor:

        batch_size, channels, height, width = pixel_values.shape

        target_dtype = self.patch_embedding.weight.dtype
        # (batch_size, hidden_size, image_size // patch_size, image_size // patch_size)
        patch_embeds = self.patch_embedding(pixel_values.to(dtype=target_dtype))

        # (batch_size, hidden_size, image_size // patch_size * image_size // patch_size) ->
        # (batch_size, image_size // patch_size * image_size // patch_size, hidden_size) =
        # (batch_size, num_patches, hidden_size)
        embeddings = patch_embeds.flatten(2).transpose(1, 2)

        if interpolate_pos_encoding:
            embeddings = embeddings + self.interpolate_pos_encoding(embeddings, height, width)
        else:
            # training with fixed image size (~num_patches), all positions are retained
            embeddings = embeddings + self.position_embedding(self.position_ids)
        return embeddings

In [8]:
class MultiheadAttentionPoolingHead(nn.Module):
    def __init__(self, config: VisionConfig) -> None:
        super().__init__()

        self.probe = nn.Parameter(torch.randn(1, 1, config.hidden_size))
        self.attention = torch.nn.MultiheadAttention(config.hidden_size, config.n_heads, batch_first=True)
        self.layernorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.mlp = MLP(config)

    def forward(self, hidden_state: torch.Tensor) -> torch.Tensor:
        # (batch, num_patches, hidden_size)
        batch_size = hidden_state.shape[0]
        # (1, 1, hidden_size) -> (batch_size, 1, hidden_size)
        probe = self.probe.repeat(batch_size, 1, 1)

        # (attn_output, attn_weights)
        hidden_state, _ = self.attention(query=probe, key=hidden_state, value=hidden_state) # (batch, 1, 768)

        residual = hidden_state
        hidden_state = self.layernorm(hidden_state)
        hidden_state = residual + self.mlp(hidden_state)

        return hidden_state[:, 0] # (batch, hidden_size)

In [ ]:
class VisionModel(nn.Module):
    def __init__(self, config: VisionConfig):
        super().__init__()

        self.config = config

        self.embeddings = VisionEmbeddings(config)
        self.encoder = Encoder(config)
        self.post_layernorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.head = MultiheadAttentionPoolingHead(config)

    def forward(
        self,
        pixel_values: torch.FloatTensor,        # preprocessed images with (batch_size, num_channels, height, width)
        interpolate_pos_encoding: bool = False, # used when fine-tuning on images of different sizes than the ones used during training
    ) -> torch.FloatTensor:

        hidden_states = self.embeddings(pixel_values, interpolate_pos_encoding)
        last_hidden_state = self.encoder(hidden_states)
        last_hidden_state = self.post_layernorm(last_hidden_state)
        # (batch_size, num_patches, hidden_size) is incompatible with the text model's output, which would be
        # a vector with shape of (batch_size, hidden_size) to repesent each sequence. Specifically,
        # from (batch_size, seq_len, hidden_size) they get the last hidden state of the last token there,
        pooler_output = self.head(last_hidden_state) # (batch_size, hidden_size)
        return last_hidden_state, pooler_output

In [ ]:
class SiglipModel(nn.Module):

    def __init__(self, text_config: TextConfig, vision_config: VisionConfig):
        super().__init__()

        self.text_config = text_config
        self.vision_config = vision_config

        self.text_model = TextModel(text_config)
        self.vision_model = VisionModel(vision_config)

        self.t_prime = nn.Parameter(torch.randn(1))
        self.b = nn.Parameter(torch.randn(1))

    def forward(
        self,
        input_ids: torch.LongTensor | None = None,
        pixel_values: torch.FloatTensor | None = None,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        interpolate_pos_encoding: bool = False,
    ):

        text_last_hidden_state, text_embeds = self.text_model(input_ids, attention_mask, position_ids)
        image_last_hidden_state, image_embeds = self.vision_model(pixel_values, interpolate_pos_encoding)
        return text_embeds, image_embeds

    def loss_fn(
        self,
        txt_emb: torch.Tensor,  # text model embedding (batch size, dim)
        img_emb: torch.Tensor, # image model embedding (batch size, dim)
    ) -> torch.Tensor:
        # sigmoid loss for language image pre-training

        # L2 normalization (p-norm = 2)
        zimg = img_emb / img_emb.norm(p=2, dim=-1, keepdim=True)
        ztxt = txt_emb / txt_emb.norm(p=2, dim=-1, keepdim=True)

        # cosine similarity as logits
        logits = torch.matmul(zimg, ztxt.t().to(text_embeds.device))

        t_prime, b = self.t_prime.to(text_embeds.device), self.b.to(text_embeds.device)

        logits = logits * t_prime.exp() + b
        eye = torch.eye(logits.size(0), device=logits.device)
        labels = 2 * eye - torch.ones_like(logits)
        loglik = torch.nn.functional.logsigmoid(labels * logits)
        nll = -torch.sum(loglik, dim=-1)
        loss = nll.mean()

        return loss

## Előtanítás

A kontrasztív előtanítás (contrastive pre-training) lényege, hogy adott $(I_j, T_j)$ kép  beágyazása és szöveg beágyazása hasonlóak legyenek a vektortérben, és minden más $(I_j, T_{i \neq j})$ rendezett pár beágyazásaitól pedig távol legyenek. Azt, hogy mit jelent a hasonló, lásd lentebb. Továbbá, azt feltételezik, hogy a rendezett párok nem függnek össze, ami a gyakorlatban sokszor nem teljesül.

Megjegyzés. A beágyazás alatt egy vektort értünk valamilyen vektortérben (a szokásos $\mathbb{T}$ test felett, ami legtöbbször a valós számok halmaza).

Ezt többféleképpen lehet elérni. A cikk két módszert részletez és mi csak a második módszert, a szigmoid-alapú veszteséget részletezzük.

Legyen $f(\cdot)$ képi model (image model) és $g(\cdot)$ szöveges model (text model), valamint legyen a képi modell kimenete $\mathbf{x}$, a szöveges modell kimenete pedig $\mathbf{y}$.

Mindkét modell kimenetén L2 normalizációt hajtanak végre, azaz

$$
\mathbf{x}_i = \frac{f\left(I_i\right)}{|| f\left(I_i\right) ||_2}
\text { és }
\mathbf{y}_i = \frac{g\left(T_i\right)}{|| g\left(T_i\right) ||_2}.
$$

A $|| \cdot ||_2$ jelenti az L2 normalizációt, ami megegyezik egy olyan p-normával, ahol $p=2$. Általánosságban a p-norma (vagy Hölder-norma) kifejezhető az alábbi alakban:

$$
\|x\|_p=\left(\sum_{i=1}^n\left|x_i\right|^p\right)^{1 / p} \quad \text{ahol } p \geq 1.
$$

A szigmoid-alapú veszteség független a batch-mérettől és kiszámítása a következőképpen történik:

$$
\begin{equation}
-\frac{1}{|B|} \sum_{i=1}^{|B|} \sum_{j=1}^{|B|}\log \frac{1}{1+e^{z_{i j}\left(-t \mathbf{x}_i \cdot \mathbf{y}_j+b\right)}}.
\end{equation}
$$

Itt $B$ a batch-méret és $z_{ij}$ az adott (kép, szöveg) párhoz tartozó címke, ami $1$, ha összetartoznak és $-1$, ha nem.


Vizsgáljuk meg a veszteségfüggvényt részletesen. A cél, hogy az eltérés kicsi legyen, ha a két vektor hasonló.

A természetes logaritmus függvény $\log_e: x \mapsto \log_e(x)$ vagy röviden $\log(x)$, és $\mathbb{R}^{+} \rightarrow \mathbb{R}$.


Azt szeretnék elérni, hogy ha két $\mathbf{x}, \mathbf{y}$ vektor hasonló, akkor a veszteség "közelebb" legyen a nullához.

De mivel a szigmoid értékkészlete $[0,1]$, ezért ez akkor fog megtörténni, ha a szigmoid közelít az $1$-hez (mert ha $1$, akkor lesz a $\log(1) = 0$). Ha közelít a $0$-hoz, akkor a $\log$ eredménye negatív lesz (például $\log(0.0001) = -9.210$).

Megjegyzés. Mivel a veszteségfüggvényben szerepel egy minusz előjel, a negatív végül pozitív lesz.

Tehát azt akarjuk, hogy a szigmoid eredménye közelítsen az 1-hez hasonló vektorok és 0-hoz különböző vektorok esetén. Ez akkor fog megtörténni, ha a szigmoid függvényben a kitevő tart a $-\infty$-hez.

Megjegyzés. Az exponenciális függvény definíciója szerint $\exp(x) = e^x$ alakú, valamint $\mathbb{R} \rightarrow \mathbb{R}^+$. A görbéje mindig pozitív és tetszőlegesen megközelíti, de soha nem érinti az x-tengelyt. Például, ha $x=-100$, akkor $e^{-100} = 3.72 \times 10^{-44}$. Természetesen, implementációs szinten nem tudunk minden valós számot ábrázolni, ezért figyelni kell a numerikus stabilitásra. Más szavakkal, a nagyon nagy és kicsi értékekre.

Vagyis hasonló párokra a kitevőnek $\left(z_{i j}\left(-t \cdot \mathbf{x}_i \cdot \mathbf{y}_j+b\right)\right)$ minél kisebbnek kell lennie két vektor hasonlósága esetén.

Nézzük végig a tagok szerepét egyesével.

A $x_i y_i$ szorzat eredménye a koszinusz távolság lesz. Ezt használják fel távolsági metrikának. A koszinusz távolság két vektor skaláris szorzatából (dot product) levezehető, ahol szorzatot a

$$
\mathbf{x y}=|\mathbf{x}| |\mathbf{y}| \cos \varphi
$$

képlet adja. Ha az egyenletet átrendezzük, a

$$
\sum_{i=1}^n a_i b_i=|\mathbf{a}||\mathbf{b}| \cos \varphi
$$

kapjuk. Mivel korábban a vektorokat normalizáltuk (L2 normalizáció), vagyis vektorhosszuk (vagy abszolútértékük) $1$, ezért a $|\mathbf{a}||\mathbf{b}|$ eltűnik az egyenletből:

$$
\cos \varphi = \sum_{i=1}^n a_i b_i.
$$

A koszinusz hasonlóság értékkészlete $[-1, 1]$. Minél hasonlóbbak a vektorok, annál közelebb lesznek az $1$-hez.

A $z_{ij}$ számról tudjuk, hogy minden egymáshoz tartozó párra $1$, és minden különböző párra $-1$.

A $b$ szám tanulható paraméter, és az eltolásért (bias) felel.

A képletben szereplő $t$ értékét az $\exp(t')=e^{t'}$ képlettel kapjuk, ahol $t'$ egy tanítható paraméter. Ez az ún. temperature (hőmérséklet).

Az inicializálás során $t = \log(10)=2.3025$ és $b=-10$.

Leegyszerűsítve, 4 fő eset áll fenn, amit végigvehetünk a veszteségfüggvény teljes megértéséhez.

Az 1. eset, hogy különbözö pár $\left(z_{i j}=-1\right)$, és $\mathbf{x}_i \cdot \mathbf{y}_j \approx 1$, tehát a modell helytelenül hasonlónak számítja. A képlet így alakul:

$$
-1 \cdot(-t \cdot 1+b)=t-b,
$$

és az eredmény pozitív, tehát a veszteség magasabb.

A 2. eset, hogy különböző pár $\left(z_{i j}=-1\right), \mathbf{x}_i \cdot \mathbf{y}_j \approx-1$, tehát a modell helyesen különbözőnek értékeli. A képlet így alakul:

$$
-1 \cdot(-t \cdot(-1)+b)=-t-b
$$

és az eredmény negatív, tehát a veszteség alacsonyabb.

A 3. eset, ha hasonló pár esetén $\left(z_{i j}=1\right)$, és $\mathbf{x}_i \cdot \mathbf{y}_j \approx-1$, tehát a modell helytelenül különbözőnek mondja. A képlet így alakul:

$$
1 \cdot(-t \cdot(-1)+b)=t+b,
$$

és az eredmény pozitív, tehát a veszteség magasabb lesz.

Az utolsó, 4. eset, ha hasonló pár $\left(z_{i j}=1\right), \mathbf{x}_i \cdot \mathbf{y}_j \approx 1$, tehát a modell helyesen hasonlónak mondja. A képlet így alakul:

$$
1 \cdot(-t \cdot 1+b)=-t+b
$$

és az eredmény negatív, tehát a veszteség alacsonyabb lesz.


Azt látjuk, hogy mind a 4 esetben a veszteségfüggvény helyesen viselkedik $t \gg b$ feltétel mellett.

A cikk egy része a tanítás és különböző hiperparaméterek optimalizálására is fókuszál, valamint összehasonlítja a softmax és szigmoid alapú veszteségfüggvényt, amiket most nem tárgyalunk.

A veszteségfüggvényt leszámítva a tanítás a szokásos gradiens leereszkedés (egy implementációja).

In [48]:
text_config, vision_config = TextConfig(), VisionConfig()

In [ ]:
model = SiglipModel(text_config, vision_config)

In [ ]:
# sanity check
text_embeds, image_embeds = model(
    batch["input_ids"].to(device),
    batch["pixel_values"].to(device)
)
loss = model.loss_fn(text_embeds,image_embeds)
print(text_embeds.shape, image_embeds.shape, loss)

torch.Size([32, 192]) torch.Size([32, 192]) tensor(25.2566, device='cuda:0', grad_fn=<MeanBackward0>)


In [ ]:
model = model.to(device)

In [ ]:
model.train()

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

In [ ]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

In [ ]:
for epoch in range(n_epochs):
    total_loss = 0.0
    num_batches = 0

    for batch in dataloader:
        if batch is None: # text can be None for some examples
            continue
        pixel_values = batch['pixel_values'].to(device)       # (bs, 3, 224, 224)
        input_ids = batch['input_ids'].to(device)             # (bs, 64)

        optimizer.zero_grad()

        image_embeds, text_embeds = model(input_ids, pixel_values)

        loss = model.loss_fn(text_embeds, image_embeds)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # gradient clipping

        optimizer.step()

        scheduler.step()

        total_loss += loss.item()
        num_batches += 1

        if num_batches % 100 == 0:
            print(f"epoch {epoch+1} | batch {num_batches} | loss {loss.item():.4f}")

    avg_loss = total_loss / num_batches
    print(f"epoch {epoch+1} complete | avg loss: {avg_loss:.4f}")

epoch 1 | batch 100 | loss 15.1861
epoch 1 | batch 200 | loss 14.3146
epoch 1 | batch 300 | loss 13.4963
epoch 1 | batch 400 | loss 12.6897
epoch 1 | batch 500 | loss 11.9182
epoch 1 | batch 600 | loss 11.1945
epoch 1 | batch 700 | loss 10.4968
epoch 1 | batch 800 | loss 9.8302
epoch 1 | batch 900 | loss 9.2238
epoch 1 complete | avg loss: 12.4102
epoch 2 | batch 100 | loss 8.5924
epoch 2 | batch 200 | loss 8.0422
epoch 2 | batch 300 | loss 7.5442
epoch 2 | batch 400 | loss 7.0847
epoch 2 | batch 500 | loss 6.6529
epoch 2 | batch 600 | loss 6.2704
epoch 2 | batch 700 | loss 5.9259
epoch 2 | batch 800 | loss 5.6151
epoch 2 | batch 900 | loss 5.3481
epoch 2 complete | avg loss: 6.9779
epoch 3 | batch 100 | loss 5.1064
epoch 3 | batch 200 | loss 4.9107
epoch 3 | batch 300 | loss 4.7529
epoch 3 | batch 400 | loss 4.6314
epoch 3 | batch 500 | loss 4.5398
epoch 3 | batch 600 | loss 4.4851
epoch 3 | batch 700 | loss 4.4634
epoch 3 | batch 800 | loss 4.4548
epoch 3 | batch 900 | loss 4.4516
ep

## Finomhangolás

A finomhangoláshoz az ImageNet 1K adathalmazt választották és azon értékelték ki a modelleket. Ezt erőforrás hiányában nem részletezzük.